In [0]:
"""
Generador de datasets para clase de procesamiento de datos.
Dominio: operaciones de campo en infraestructura energética.

Tres entidades relacionadas con ruido intencional:
  - equipos              : activos físicos desplegados en campo
  - ordenes_trabajo      : órdenes de mantenimiento/intervención
  - registros_telemetria : lecturas de sensores en equipos de campo

Reproducible con seed fijo.
Retorna listas de dicts. El caller decide cómo parsear (Spark, pandas, etc).
"""

import random
from datetime import datetime, timedelta
from faker import Faker

fake = Faker("es_AR")
Faker.seed(42)
random.seed(42)

# ---------------------------------------------------------------------------
# Constantes de dominio
# ---------------------------------------------------------------------------

REGIONES_LIMPIAS = ["NOA", "NEA", "CUYO", "PATAGONIA", "PAMPEANA", "GBA"]

# Ruido intencional en región: variantes sucias que habrá que normalizar
REGIONES_CON_RUIDO = REGIONES_LIMPIAS + [
    "noa", "Noa", "N.O.A", "NEA ", " NEA", "Nea",
    "cuyo", "CUYO ", "Patagonia", "patagonia", "PATAGONÍA",
    "pampeana", "Pampeana", "GBA ", "gba", "G.B.A",
    None, None,  # nulos reales
]

TIPOS_EQUIPO = ["transformador", "generador", "interruptor", "medidor", "rectificador"]

ESTADOS_EQUIPO_LIMPIOS = ["operativo", "en_mantenimiento", "fuera_de_servicio"]
ESTADOS_EQUIPO_CON_RUIDO = ESTADOS_EQUIPO_LIMPIOS + [
    "Operativo", "OPERATIVO", "op", "operativo ",
    "En mantenimiento", "en mantenimiento", "mant",
    "fuera de servicio", "FDS", "fds",
    None,
]

TIPOS_EVENTO = ["alerta_tension", "alerta_temperatura", "falla_comunicacion",
                "lectura_normal", "corte_programado", "sobrecarga"]

PRIORIDADES = ["P1", "P2", "P3"]
PRIORIDADES_CON_RUIDO = PRIORIDADES + ["p1", "p 1", "P-1", "P1 ", "p2", "P 2", "p3", None]

TECNICOS_IDS = [f"TEC-{str(i).zfill(3)}" for i in range(1, 31)]


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def fecha_aleatoria(inicio: datetime, fin: datetime) -> datetime:
    delta = fin - inicio
    segundos = random.randint(0, int(delta.total_seconds()))
    return inicio + timedelta(seconds=segundos)


def formato_fecha_con_ruido(dt: datetime) -> str:
    """Retorna la fecha en uno de varios formatos, para simular fuentes heterogéneas."""
    formatos = [
        "%Y-%m-%d %H:%M:%S",   # canónico
        "%Y-%m-%d %H:%M:%S",   # canónico repetido para darle más peso
        "%Y-%m-%dT%H:%M:%S",   # ISO sin zona
        "%d/%m/%Y %H:%M",      # argentino clásico
        "%d-%m-%Y",            # sin hora
        "%Y%m%d%H%M%S",        # compacto sin separadores
    ]
    return dt.strftime(random.choice(formatos))


# ---------------------------------------------------------------------------
# Dataset 1: equipos
# ---------------------------------------------------------------------------

def generar_equipos(n: int = 200) -> list:
    """
    Activos físicos en campo.

    Ruido plantado:
      - region: variantes sucias y nulos
      - estado: casing inconsistente y abreviaciones
      - fecha_instalacion: formatos mixtos
      - voltaje_nominal: algunos negativos (error de carga)
      - fabricante: algunos nulos
    """
    random.seed(42)
    registros = []

    for i in range(1, n + 1):
        equipo_id = f"EQ-{str(i).zfill(4)}"
        tipo = random.choice(TIPOS_EQUIPO)

        fecha_inst = fecha_aleatoria(
            datetime(2010, 1, 1),
            datetime(2022, 12, 31)
        )

        # Ruido en fecha_instalacion: 15% formato no canónico
        fecha_str = (
            formato_fecha_con_ruido(fecha_inst)
            if random.random() < 0.15
            else fecha_inst.strftime("%Y-%m-%d %H:%M:%S")
        )

        # Ruido en voltaje_nominal: 5% negativos
        voltaje = random.choice([13.8, 33.0, 66.0, 132.0, 220.0])
        if random.random() < 0.05:
            voltaje = -voltaje

        # Ruido en fabricante: 8% nulos
        fabricante = (
            random.choice(["ABB", "Siemens", "Schneider", "GE", "Eaton"])
            if random.random() > 0.08 else None
        )

        registros.append({
            "equipo_id":          equipo_id,
            "tipo":               tipo,
            "region":             random.choice(REGIONES_CON_RUIDO),
            "estado":             random.choice(ESTADOS_EQUIPO_CON_RUIDO),
            "voltaje_nominal_kv": voltaje,
            "fabricante":         fabricante,
            "fecha_instalacion":  fecha_str,
            "cliente_id":         f"CLI-{str(random.randint(1, 40)).zfill(3)}",
        })

    return registros


# ---------------------------------------------------------------------------
# Dataset 2: ordenes_trabajo
# ---------------------------------------------------------------------------

def generar_ordenes_trabajo(n: int = 400) -> list:
    """
    Órdenes de intervención/mantenimiento sobre equipos.

    Ruido plantado:
      - prioridad: casing inconsistente y nulos
      - fecha_cierre: nula cuando la orden sigue abierta (legítimo),
        pero también en órdenes que deberían estar cerradas (problema)
      - costo_estimado: algunos negativos o cero
      - tecnico_asignado: 10% nulos (orden creada sin asignar)
      - duplicados: ~3% de registros son reenvíos del mismo orden_id
    """
    random.seed(42)
    inicio_periodo = datetime(2020, 1, 1)
    fin_periodo    = datetime(2024, 6, 30)

    equipo_ids = [f"EQ-{str(i).zfill(4)}" for i in range(1, 201)]
    registros  = []

    orden_id = 1
    while len(registros) < n:
        oid = f"OT-{str(orden_id).zfill(5)}"
        orden_id += 1

        fecha_apertura = fecha_aleatoria(inicio_periodo, fin_periodo)

        # 20% de órdenes sin fecha_cierre (abiertas o error de registro)
        if random.random() > 0.20:
            dias_duracion  = random.randint(1, 60)
            fecha_cierre   = fecha_apertura + timedelta(days=dias_duracion)
            fecha_cierre_s = fecha_cierre.strftime("%Y-%m-%d %H:%M:%S")
        else:
            fecha_cierre_s = None

        costo = round(random.uniform(500, 80000), 2)
        # Ruido: 4% costos negativos
        if random.random() < 0.04:
            costo = -costo
        # Ruido: 3% costo cero
        elif random.random() < 0.03:
            costo = 0.0

        registro = {
            "orden_id":          oid,
            "equipo_id":         random.choice(equipo_ids),
            "tipo_trabajo":      random.choice(["correctivo", "preventivo", "predictivo", "emergencia"]),
            "prioridad":         random.choice(PRIORIDADES_CON_RUIDO),
            "tecnico_asignado":  random.choice(TECNICOS_IDS) if random.random() > 0.10 else None,
            "fecha_apertura":    fecha_apertura.strftime("%Y-%m-%d %H:%M:%S"),
            "fecha_cierre":      fecha_cierre_s,
            "costo_estimado":    costo,
            "resolucion":        random.choice(["exitosa", "parcial", "sin_resolucion", None]),
        }
        registros.append(registro)

        # Duplicados: ~3% de probabilidad de reenviar el mismo registro
        if random.random() < 0.03:
            registros.append(registro.copy())

    return registros[:n]


# ---------------------------------------------------------------------------
# Dataset 3: registros_telemetria
# ---------------------------------------------------------------------------

def generar_telemetria(n: int = 2000) -> list:
    """
    Lecturas de sensores asociadas a equipos.
    Dataset de mayor volumen; es el fact table del pipeline.

    Ruido plantado:
      - equipo_id: 6% nulos (sensor no identificó el origen)
      - timestamp: formatos mixtos
      - valor_medido: outliers negativos y valores extremos
      - duplicados: ~5% reenvíos del mismo evento
      - tipo_evento: algunos valores libres/mal escritos
      - unidad_medida: inconsistencia (kV / KV / kilovolt)
    """
    random.seed(42)
    inicio_periodo = datetime(2022, 1, 1)
    fin_periodo    = datetime(2024, 6, 30)

    equipo_ids = [f"EQ-{str(i).zfill(4)}" for i in range(1, 201)]

    # Ruido en tipo_evento
    tipos_con_ruido = TIPOS_EVENTO + [
        "Alerta_Tension", "ALERTA TEMPERATURA", "falla comunicacion",
        "lectura normal", "Sobrecarga", "SOBRECARGA",
    ]

    # Ruido en unidad_medida
    unidades_con_ruido = ["kV", "kV", "KV", "kilovolt", "Kv", "kv", "A", "A", "amp", "AMP"]

    registros = []
    evento_id = 1

    while len(registros) < n:
        eid = f"EVT-{str(evento_id).zfill(6)}"
        evento_id += 1

        ts = fecha_aleatoria(inicio_periodo, fin_periodo)
        ts_str = (
            formato_fecha_con_ruido(ts)
            if random.random() < 0.20
            else ts.strftime("%Y-%m-%d %H:%M:%S")
        )

        valor = round(random.uniform(0.5, 300.0), 3)
        # Ruido: 4% negativos
        if random.random() < 0.04:
            valor = -valor
        # Ruido: 2% valores extremos (sensor desconectado reporta 9999)
        elif random.random() < 0.02:
            valor = 9999.0

        registro = {
            "evento_id":      eid,
            "equipo_id":      random.choice(equipo_ids) if random.random() > 0.06 else None,
            "timestamp":      ts_str,
            "tipo_evento":    random.choice(tipos_con_ruido),
            "valor_medido":   valor,
            "unidad_medida":  random.choice(unidades_con_ruido),
            "sensor_id":      f"SEN-{str(random.randint(1, 500)).zfill(4)}",
            "calidad_senal":  random.choice(["buena", "degradada", "sin_senal", None]),
        }
        registros.append(registro)

        # Duplicados: ~5% reenvío
        if random.random() < 0.05:
            registros.append(registro.copy())

    return registros[:n]